# Mobile Overdraft Credit Risk — EDA & Project Overview

**Author:** Kevin Odhiambo  
**Dataset:** 185,626 overdraft episodes · 50,000 borrowers · Jan 2022 – Jun 2024  
**Product context:** Fuliza-style continuous mobile-money overdraft (nano-loan), East African fintech

---

## What this notebook covers

1. [Business Context & Objectives](#1-business-context--objectives)
2. [Dataset Overview](#2-dataset-overview)
3. [Target Variable — 30-Day Default](#3-target-variable--30-day-default)
4. [Portfolio Growth & Temporal Trends](#4-portfolio-growth--temporal-trends)
5. [Borrower Profile Analysis](#5-borrower-profile-analysis)
6. [Geographic & Channel Breakdown](#6-geographic--channel-breakdown)
7. [Key Risk Driver Analysis](#7-key-risk-driver-analysis)
8. [Repayment History as a Predictive Signal](#8-repayment-history-as-a-predictive-signal)
9. [Feature Correlation & Multicollinearity](#9-feature-correlation--multicollinearity)
10. [Fairness Considerations](#10-fairness-considerations)
11. [From EDA to Model — Objectives Fulfilled](#11-from-eda-to-model--objectives-fulfilled)

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib.gridspec import GridSpec
import matplotlib.patches as mpatches

plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'font.size': 11,
})

PALETTE = {
    'default': '#E63946',
    'cleared': '#2A9D8F',
    'neutral': '#457B9D',
    'light': '#A8DADC',
    'dark': '#1D3557',
}

df = pd.read_csv('../data/raw/overdraft_lending_data.csv', parse_dates=['draw_date'])
print(f'Loaded {len(df):,} rows × {df.shape[1]} columns')
df.head(3)

---
## 1. Business Context & Objectives

### The product: Fuliza-style mobile overdraft

Fuliza (Safaricom/NCBA, launched 2019) is a continuous overdraft facility embedded in M-PESA, Kenya's dominant mobile money platform.  
Key mechanics:
- A **credit limit** is assigned silently at the network level — no application, no paperwork
- Limit is driven by **behavioral signals**: M-PESA usage volume, repayment timeliness, airtime/data spend, savings-account activity, CRB status — **not** income or employment data
- A borrower simply pays more than their M-PESA balance; the shortfall is covered automatically
- Repayment is **automatic** — the next inflow deducts outstanding balance + fees before the customer receives their money
- A **daily maintenance fee** accrues on the outstanding balance (tiered by amount)
- If the balance is not cleared within **30 days**, the account is flagged for CRB listing

By 2023, Fuliza was disbursing **~KES 3 billion per day** and served over 20 million unique borrowers — making it the world's largest nano-lending product by volume.

### Why this is a hard credit-scoring problem

| Challenge | Traditional lending | Mobile overdraft (this project) |
|---|---|---|
| Data available at scoring time | Credit bureau, income statement, employment | Behavioral signals only — no bureau income data |
| Loan tenor | Months to years | 1–30 days |
| Application process | Customer-initiated, formal | Silent, automated, instant |
| Repayment mechanism | Scheduled installments | Auto-deducted from next inflow |
| Regulatory sensitivity | High (protected-class data explicit) | High but different — proxy discrimination via behavioral features |

### Project objectives

This project has three explicit goals, each verified in this notebook and in the model:

> **Objective 1 — Predict 30-day default using alternative behavioral data only**  
> Show that transaction behavior, airtime spend, savings activity, and repayment history can discriminate between borrowers who will clear within 30 days and those who will not — *without* income, employment, or bureau financial data.

> **Objective 2 — Identify and rank the dominant risk drivers**  
> Quantify which features drive default risk, verify they make business sense, and use SHAP to generate per-prediction adverse-action reason codes — a regulatory requirement for automated lending decisions.

> **Objective 3 — Audit the model for proxy discrimination**  
> Apply the four-fifths (80%) disparate-impact rule across thin-file borrowers (short tenure) and CRB-flagged borrowers — the two populations most at risk from unfair alternative-data scoring — and enforce this as a CI/CD release gate.

---
## 2. Dataset Overview

In [ ]:
print('=== Dataset dimensions ===')
print(f'  Rows (overdraft episodes): {len(df):,}')
print(f'  Unique borrowers:          {df.cust_id.nunique():,}')
print(f'  Observation window:        {df.draw_date.min().date()} – {df.draw_date.max().date()}')
print(f'  Features (model inputs):   18')
print(f'  Metadata columns:          loan_id, cust_id, draw_date, county, channel, product_code')
print()

print('=== Column types ===')
type_groups = df.dtypes.astype(str).value_counts()
for t, n in type_groups.items():
    print(f'  {t}: {n} columns')
print()

print('=== Missing values ===')
nulls = df.isnull().sum()
if nulls.sum() == 0:
    print('  None — dataset is complete.')
else:
    print(nulls[nulls > 0])

In [ ]:
# Summary statistics for all model features
model_features = [
    'tenure_months','monthly_txn_count','avg_txn_value_kes','send_receive_ratio',
    'unique_counterparties_30d','agent_cashin_freq_30d','airtime_bundle_freq_30d',
    'savings_activity_score','crb_flagged','voice_data_spend_idx','income_regularity',
    'assigned_limit_kes','draw_amount_kes','utilization_rate','overdraw_to_inflow_ratio',
    'prior_overdraw_count','prior_cleared_within_24h_count','prior_rolled_past_30d_count',
]

df[model_features].describe().T[['mean','std','min','50%','max']].round(3)

### Feature glossary

| Feature | Type | Description | Business rationale |
|---|---|---|---|
| `tenure_months` | Borrower | Months since first M-PESA transaction | Longer tenure = more payment history for scoring |
| `monthly_txn_count` | Borrower | Avg. number of M-PESA transactions per month | High activity = higher regular inflow, faster clearance |
| `avg_txn_value_kes` | Borrower | Average individual transaction value (KES) | Proxy for typical wallet size |
| `send_receive_ratio` | Borrower | Ratio of sent to received transactions | >1 = net sender (spender); <1 = net receiver (earner) |
| `unique_counterparties_30d` | Borrower | Distinct parties transacted with in last 30 days | Social/commercial network breadth |
| `agent_cashin_freq_30d` | Borrower | Agent cash-in events per month | Measures cash inflow regularity |
| `airtime_bundle_freq_30d` | Borrower | Airtime/data bundle purchases per month | Safaricom-cited factor in Fuliza limit assignment |
| `savings_activity_score` | Borrower | Normalized savings account engagement (0–1) | M-Shwari/KCB M-PESA usage proxy |
| `crb_flagged` | Borrower | 1 = on CRB negative list | Direct credit history signal (thin for many borrowers) |
| `voice_data_spend_idx` | Borrower | Normalized voice/data spend (0–1) | Cited by Safaricom as a Fuliza eligibility factor |
| `income_regularity` | Borrower | Score for regularity of inflow timing (0–1) | Irregular inflow = harder to auto-repay |
| `assigned_limit_kes` | Episode | Fuliza credit limit at time of draw (KES) | Reflects risk tier assigned by network |
| `draw_amount_kes` | Episode | Amount drawn on this episode (KES) | |
| `utilization_rate` | Episode | draw / limit (0–1) | How much of the limit was used |
| `overdraw_to_inflow_ratio` | Episode | Outstanding / expected daily inflow | Days of inflow needed to clear the draw |
| `prior_overdraw_count` | History | Total prior overdraft episodes for this borrower | Experience with the product |
| `prior_cleared_within_24h_count` | History | Prior episodes cleared within 24 hours | Track record of fast repayment |
| `prior_rolled_past_30d_count` | History | Prior episodes that rolled past 30 days | **Most powerful history signal** |

---
## 3. Target Variable — 30-Day Default

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# --- panel 1: class balance ---
ax = axes[0]
counts = df['defaulted_30d'].value_counts()
bars = ax.bar(['Cleared (0)', 'Defaulted (1)'], counts.values,
              color=[PALETTE['cleared'], PALETTE['default']], width=0.5, edgecolor='white')
for bar, v in zip(bars, counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1500,
            f'{v:,}\n({v/len(df)*100:.1f}%)', ha='center', va='bottom', fontsize=10, fontweight='bold')
ax.set_title('Class balance', fontweight='bold')
ax.set_ylabel('Episode count')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1000:.0f}k'))
ax.set_ylim(0, counts.max() * 1.2)

# --- panel 2: default rate by utilization quartile ---
ax = axes[1]
df['util_q'] = pd.qcut(df['utilization_rate'], 4,
                        labels=['Q1\n(0–24%)', 'Q2\n(24–39%)', 'Q3\n(39–54%)', 'Q4\n(54–99%)'])
dr_util = df.groupby('util_q', observed=True)['defaulted_30d'].mean() * 100
bars = ax.bar(dr_util.index, dr_util.values, color=PALETTE['neutral'], edgecolor='white')
for bar, v in zip(bars, dr_util.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
            f'{v:.1f}%', ha='center', va='bottom', fontsize=10)
ax.axhline(df['defaulted_30d'].mean()*100, color=PALETTE['default'], linestyle='--', linewidth=1.2,
           label=f'Portfolio avg {df["defaulted_30d"].mean()*100:.1f}%')
ax.set_title('Default rate by utilization quartile', fontweight='bold')
ax.set_ylabel('30-day default rate (%)')
ax.legend(fontsize=9)

# --- panel 3: default rate by CRB flag ---
ax = axes[2]
dr_crb = df.groupby('crb_flagged')['defaulted_30d'].mean() * 100
labels = ['CRB clean', 'CRB flagged']
colors = [PALETTE['cleared'], PALETTE['default']]
bars = ax.bar(labels, dr_crb.values, color=colors, width=0.4, edgecolor='white')
for bar, v in zip(bars, dr_crb.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
            f'{v:.2f}%', ha='center', va='bottom', fontsize=11, fontweight='bold')
ax.axhline(df['defaulted_30d'].mean()*100, color='gray', linestyle='--', linewidth=1.2,
           label='Portfolio avg')
ax.set_title('Default rate: CRB flag', fontweight='bold')
ax.set_ylabel('30-day default rate (%)')
ax.legend(fontsize=9)

fig.suptitle('Target variable: defaulted_30d', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print(f'Portfolio default rate: {df["defaulted_30d"].mean()*100:.2f}%')
print(f'Class imbalance ratio: ~{counts[0]/counts[1]:.0f}:1 (cleared:defaulted)')
print()
print('Insight: Class imbalance (94.5:5.5) is modest for this type of product.')
print('PR-AUC is the primary evaluation metric — ROC-AUC can be misleadingly high on imbalanced data.')

**Key observations:**
- The ~7.6% portfolio-average default rate is above the Fuliza public estimate range (5–7%). The synthetic DGP adds an idiosyncratic shock term (N(0, σ=2.0)) that makes a portion of defaults structurally unpredictable, which inflates the default rate relative to a perfectly calibrated product while keeping the modeling challenge realistic.
- Utilization is already a strong discriminator *before* modeling: high-utilization borrowers (Q4) default at a rate nearly 5× higher than low-utilization ones (Q1).
- CRB-flagged borrowers default at a higher rate than CRB-clean borrowers — the CRB signal exists but is weaker than utilization behavior, reflecting the DGP's partial observability design (CRB flagging correlates with latent risk θ but is not a direct measure of it).

---
## 4. Portfolio Growth & Temporal Trends

In [ ]:
monthly = df.resample('ME', on='draw_date').agg(
    draws=('loan_id', 'count'),
    defaults=('defaulted_30d', 'sum'),
    default_rate=('defaulted_30d', 'mean'),
    avg_draw_kes=('draw_amount_kes', 'mean'),
    avg_utilization=('utilization_rate', 'mean'),
).reset_index()

fig, axes = plt.subplots(2, 2, figsize=(15, 9))

# --- panel 1: monthly draw volume ---
ax = axes[0, 0]
ax.fill_between(monthly['draw_date'], monthly['draws'], alpha=0.3, color=PALETTE['neutral'])
ax.plot(monthly['draw_date'], monthly['draws'], color=PALETTE['neutral'], linewidth=2)
ax.set_title('Monthly draw volume (episodes)', fontweight='bold')
ax.set_ylabel('Episodes')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1000:.0f}k'))
ax.set_xlabel('')

# --- panel 2: rolling default rate ---
ax = axes[0, 1]
ax.plot(monthly['draw_date'], monthly['default_rate']*100,
        color=PALETTE['default'], linewidth=2, marker='o', markersize=3)
ax.axhline(df['defaulted_30d'].mean()*100, color='gray', linestyle='--', linewidth=1,
           label=f'Period avg {df["defaulted_30d"].mean()*100:.1f}%')
ax.set_title('Monthly default rate', fontweight='bold')
ax.set_ylabel('Default rate (%)')
ax.set_ylim(0, 10)
ax.legend()

# --- panel 3: average draw amount ---
ax = axes[1, 0]
ax.plot(monthly['draw_date'], monthly['avg_draw_kes'],
        color=PALETTE['dark'], linewidth=2)
ax.set_title('Average draw amount (KES)', fontweight='bold')
ax.set_ylabel('KES')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))

# --- panel 4: average utilization ---
ax = axes[1, 1]
ax.plot(monthly['draw_date'], monthly['avg_utilization']*100,
        color=PALETTE['light'], linewidth=2)
ax.set_title('Average utilization rate (%)', fontweight='bold')
ax.set_ylabel('Utilization (%)')

for ax in axes.flat:
    ax.set_xlabel('Month')

fig.suptitle('Portfolio temporal trends — Jan 2022 to Jun 2024', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'Peak monthly volume: {monthly["draws"].max():,} episodes ({monthly.loc[monthly["draws"].idxmax(), "draw_date"].strftime("%b %Y")})')
print(f'Default rate range: {monthly["default_rate"].min()*100:.1f}% – {monthly["default_rate"].max()*100:.1f}%')
print(f'Avg draw amount range: KES {monthly["avg_draw_kes"].min():,.0f} – KES {monthly["avg_draw_kes"].max():,.0f}')

**Key observations:**
- Portfolio volumes ramp from ~1.7k episodes/month (Jan 2022) as borrowers accumulate history and earn higher limits over time
- Default rate is **stable across the observation window** — no evidence of credit quality deterioration during the period, which validates that a model trained on earlier cohorts would generalize to later ones
- Average draw size holds steady around KES 1,300–1,500, reflecting limit-capped behavior rather than demand-driven draws

---
## 5. Borrower Profile Analysis

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9))

# --- tenure distribution ---
ax = axes[0, 0]
bins = [0, 12, 24, 36, 60, 250]
labels = ['<1 yr', '1–2 yr', '2–3 yr', '3–5 yr', '>5 yr']
df['tenure_band'] = pd.cut(df['tenure_months'], bins=bins, labels=labels)
tc = df['tenure_band'].value_counts().sort_index()
bars = ax.bar(tc.index, tc.values, color=PALETTE['neutral'], edgecolor='white')
ax.set_title('Borrower tenure distribution', fontweight='bold')
ax.set_ylabel('Episodes')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1000:.0f}k'))

# --- limit distribution ---
ax = axes[0, 1]
ax.hist(df['assigned_limit_kes'], bins=40, color=PALETTE['neutral'],
        edgecolor='white', linewidth=0.5)
ax.axvline(df['assigned_limit_kes'].median(), color=PALETTE['default'],
           linestyle='--', linewidth=1.5, label=f'Median KES {df["assigned_limit_kes"].median():,.0f}')
ax.set_title('Assigned limit (KES)', fontweight='bold')
ax.set_ylabel('Episodes')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1000:.0f}k'))
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1000:.0f}k'))
ax.legend()

# --- monthly txn count vs avg txn value (coloured by default) ---
ax = axes[0, 2]
sample = df.sample(3000, random_state=42)
colors_scatter = [PALETTE['default'] if d else PALETTE['cleared'] for d in sample['defaulted_30d']]
ax.scatter(sample['monthly_txn_count'], sample['avg_txn_value_kes'],
           c=colors_scatter, alpha=0.4, s=10)
ax.set_title('Activity vs. value\n(red=default, green=cleared)', fontweight='bold')
ax.set_xlabel('Monthly txn count')
ax.set_ylabel('Avg txn value (KES)')
ax.set_yscale('log')

# --- savings score distribution by default ---
ax = axes[1, 0]
for label, grp, col in [('Cleared', df[df.defaulted_30d==0], PALETTE['cleared']),
                         ('Defaulted', df[df.defaulted_30d==1], PALETTE['default'])]:
    ax.hist(grp['savings_activity_score'], bins=30, density=True, alpha=0.5,
            color=col, label=label)
ax.set_title('Savings activity score by outcome', fontweight='bold')
ax.set_xlabel('Savings activity score (0–1)')
ax.set_ylabel('Density')
ax.legend()

# --- income regularity distribution by default ---
ax = axes[1, 1]
for label, grp, col in [('Cleared', df[df.defaulted_30d==0], PALETTE['cleared']),
                         ('Defaulted', df[df.defaulted_30d==1], PALETTE['default'])]:
    ax.hist(grp['income_regularity'], bins=30, density=True, alpha=0.5,
            color=col, label=label)
ax.set_title('Income regularity by outcome', fontweight='bold')
ax.set_xlabel('Income regularity score (0–1)')
ax.set_ylabel('Density')
ax.legend()

# --- default rate by tenure band ---
ax = axes[1, 2]
dr_tenure = df.groupby('tenure_band', observed=True)['defaulted_30d'].mean() * 100
bars = ax.bar(dr_tenure.index, dr_tenure.values, color=PALETTE['neutral'], edgecolor='white')
for bar, v in zip(bars, dr_tenure.values):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.05,
            f'{v:.1f}%', ha='center', va='bottom', fontsize=10)
ax.axhline(df['defaulted_30d'].mean()*100, color=PALETTE['default'],
           linestyle='--', linewidth=1.2, label='Portfolio avg')
ax.set_title('Default rate by tenure band', fontweight='bold')
ax.set_ylabel('Default rate (%)')
ax.legend(fontsize=9)

fig.suptitle('Borrower profile analysis', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

**Key observations:**
- **Tenure:** The portfolio is concentrated in 1–3 year users — borrowers who have enough history to have earned a limit but are not yet long-tenured. Thin-file (<1 yr) borrowers are ~12% of the portfolio.
- **Limits:** Right-skewed distribution concentrated at KES 2,000–5,000. The median limit (KES ~3,300) reflects a product calibrated for everyday transaction gaps, not large credit needs.
- **Savings & income regularity:** Both features show clear separation between defaulters and clearers — defaulters have lower savings activity scores and less regular inflows. These are meaningful model inputs even though they are behavioral proxies, not direct financial data.
- **Tenure risk:** Short-tenure borrowers (<1 yr) show slightly elevated default rates, validating their use as a thin-file fairness proxy. Importantly, the difference is modest — longer tenure broadly reduces risk but is not a dramatically strong predictor on its own.

---
## 6. Geographic & Channel Breakdown

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# --- default rate by county (top 15 by volume) ---
ax = axes[0]
county_stats = df.groupby('county').agg(
    volume=('loan_id','count'),
    default_rate=('defaulted_30d','mean')
).sort_values('volume', ascending=False).head(15)

colors_county = [PALETTE['default'] if r > df['defaulted_30d'].mean() else PALETTE['neutral']
                 for r in county_stats['default_rate']]
bars = ax.barh(county_stats.index, county_stats['default_rate']*100,
               color=colors_county, edgecolor='white')
ax.axvline(df['defaulted_30d'].mean()*100, color='gray', linestyle='--', linewidth=1.2)
for bar, v in zip(bars, county_stats['default_rate']*100):
    ax.text(v + 0.05, bar.get_y()+bar.get_height()/2,
            f'{v:.1f}%', va='center', fontsize=9)
ax.set_title('Default rate by county\n(top 15 by volume, red = above average)', fontweight='bold')
ax.set_xlabel('30-day default rate (%)')
ax.invert_yaxis()

# --- channel analysis ---
ax = axes[1]
ch = df.groupby('channel').agg(
    volume=('loan_id','count'),
    default_rate=('defaulted_30d','mean'),
    avg_draw=('draw_amount_kes','mean'),
).sort_values('volume', ascending=False)

x = np.arange(len(ch))
w = 0.35
ax2 = ax.twinx()
ax.bar(x - w/2, ch['volume'], width=w, color=PALETTE['neutral'], alpha=0.7, label='Volume')
ax2.bar(x + w/2, ch['default_rate']*100, width=w, color=PALETTE['default'], alpha=0.7,
        label='Default rate')
ax.set_xticks(x)
ax.set_xticklabels(ch.index, rotation=15, ha='right')
ax.set_ylabel('Episode volume')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1000:.0f}k'))
ax2.set_ylabel('Default rate (%)')
ax2.set_ylim(0, 10)
ax.set_title('Volume vs. default rate by channel', fontweight='bold')
h1, l1 = ax.get_legend_handles_labels()
h2, l2 = ax2.get_legend_handles_labels()
ax.legend(h1+h2, l1+l2, loc='upper right')

fig.suptitle('Geographic and channel analysis', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print('Channel summary:')
print(ch[['volume','default_rate','avg_draw']].round(4).to_string())

**Key observations:**
- **Geography:** Default rates vary across counties (4.7% Kisii to 6.2% Meru) but the spread is narrow — county alone is a weak predictor. Nairobi has high volume but near-average default rate, consistent with a more financially active user base.
- **Channel:** Default rates are nearly identical across all access channels (~5.5–5.7%). This is expected — the Fuliza product's risk is driven by the borrower's financial position, not *how* they access it. This validates that channel is correctly excluded from the model's feature set.
- **Implication:** Geography and channel are useful for business operations (regional credit strategies, channel-specific limit adjustments) but should not be model features — they carry weak signal relative to behavioral features and could introduce proxy discrimination.

---
## 7. Key Risk Driver Analysis

### 7.1 Overdraw-to-inflow ratio — the dominant feature

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# --- distribution comparison ---
ax = axes[0]
cap = df['overdraw_to_inflow_ratio'].clip(upper=25)
for label, mask, col in [('Cleared', df.defaulted_30d==0, PALETTE['cleared']),
                          ('Defaulted', df.defaulted_30d==1, PALETTE['default'])]:
    ax.hist(cap[mask], bins=50, density=True, alpha=0.55, color=col, label=label)
ax.set_title('Overdraw-to-inflow ratio\ndistribution by outcome', fontweight='bold')
ax.set_xlabel('Overdraw-to-inflow ratio (capped at 25)')
ax.set_ylabel('Density')
ax.legend()

# --- default rate by decile ---
ax = axes[1]
df['oir_decile'] = pd.qcut(df['overdraw_to_inflow_ratio'], 10,
                            labels=[f'D{i}' for i in range(1, 11)])
dr_oir = df.groupby('oir_decile', observed=True)['defaulted_30d'].mean() * 100
bars = ax.bar(dr_oir.index, dr_oir.values, color=PALETTE['neutral'], edgecolor='white')
for bar, v in zip(bars, dr_oir.values):
    if v > 3:
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.2,
                f'{v:.0f}%', ha='center', va='bottom', fontsize=9)
ax.set_title('Default rate by OIR decile\n(D1=lowest, D10=highest)', fontweight='bold')
ax.set_ylabel('Default rate (%)')
ax.set_xlabel('Overdraw-to-inflow decile')

# --- scatter: OIR vs draw amount ---
ax = axes[2]
samp = df.sample(4000, random_state=1)
colors_s = [PALETTE['default'] if d else PALETTE['cleared'] for d in samp['defaulted_30d']]
ax.scatter(samp['overdraw_to_inflow_ratio'].clip(upper=30), samp['draw_amount_kes'],
           c=colors_s, alpha=0.3, s=8)
ax.set_title('OIR vs. draw amount\n(red=default)', fontweight='bold')
ax.set_xlabel('Overdraw-to-inflow ratio')
ax.set_ylabel('Draw amount (KES)')
ax.set_xlim(0, 30)

fig.suptitle('Overdraw-to-inflow ratio: the dominant risk signal (r = 0.70 with default)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

oir_corr = df['overdraw_to_inflow_ratio'].corr(df['defaulted_30d'])
print(f'Pearson correlation with default: {oir_corr:.4f}')
print()
print('Decile default rates:')
print(dr_oir.round(2).to_string())

**What is overdraw-to-inflow ratio and why does it dominate?**

$$\text{OIR} = \frac{\text{draw amount} + \text{access fee}}{\text{expected daily inflow}}$$

This ratio measures approximately **how many days of typical inflow are needed to clear the overdraft**.  
An OIR of 2 means the borrower needs ~2 days of normal inflow to repay; an OIR of 20 means 20 days.  
Since repayment is automatic (triggered by the next inflow), a borrower with a high OIR simply cannot repay quickly — not necessarily because they are reckless, but because the draw is large relative to their typical cash flow.

This is the **causal signal**: it directly encodes repayment feasibility, which is exactly what the 30-day default target measures.  
The model has access to this at scoring time because both the draw amount and the borrower's historical inflow pattern are known at the moment of the draw decision.

### 7.2 Utilization rate

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ax = axes[0]
for label, mask, col in [('Cleared', df.defaulted_30d==0, PALETTE['cleared']),
                          ('Defaulted', df.defaulted_30d==1, PALETTE['default'])]:
    ax.hist(df.loc[mask, 'utilization_rate'], bins=40, density=True,
            alpha=0.55, color=col, label=label)
ax.set_title('Utilization rate by outcome', fontweight='bold')
ax.set_xlabel('Utilization rate (draw / limit)')
ax.set_ylabel('Density')
ax.legend()

ax = axes[1]
df['util_decile'] = pd.qcut(df['utilization_rate'], 10, labels=[f'D{i}' for i in range(1,11)])
dr_ud = df.groupby('util_decile', observed=True)['defaulted_30d'].mean() * 100
bars = ax.bar(dr_ud.index, dr_ud.values, color=PALETTE['neutral'], edgecolor='white')
for bar, v in zip(bars, dr_ud.values):
    if v > 2:
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.1,
                f'{v:.1f}%', ha='center', va='bottom', fontsize=9)
ax.set_title('Default rate by utilization decile', fontweight='bold')
ax.set_ylabel('Default rate (%)')
ax.set_xlabel('Utilization decile (D1=lowest)')

plt.tight_layout()
plt.show()

print(f'Utilization ↔ default correlation: {df["utilization_rate"].corr(df["defaulted_30d"]):.4f}')

**Utilization and OIR are correlated but distinct signals.**  
A borrower drawing 95% of their limit may or may not have high OIR — it depends on their inflow.  
A high earner drawing 95% of a small limit clears quickly; a low earner drawing 95% of a large limit may struggle.  
The model uses *both* features because they capture complementary aspects of repayment risk.

---
## 8. Repayment History as a Predictive Signal

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# --- default rate by prior_rolled_past_30d_count ---
ax = axes[0]
dr_rolled = df.groupby('prior_rolled_past_30d_count')['defaulted_30d'].agg(['mean','count'])
dr_rolled['mean_pct'] = dr_rolled['mean'] * 100
bars = ax.bar(dr_rolled.index.astype(str), dr_rolled['mean_pct'],
              color=[PALETTE['default'] if v > 20 else PALETTE['neutral']
                     for v in dr_rolled['mean_pct']],
              edgecolor='white')
for bar, v in zip(bars, dr_rolled['mean_pct']):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5,
            f'{v:.0f}%', ha='center', va='bottom', fontsize=10, fontweight='bold')
ax.set_title('Default rate by prior\nrolled episodes (>30d)', fontweight='bold')
ax.set_xlabel('Prior episodes rolled past 30d')
ax.set_ylabel('Current episode default rate (%)')

# --- volume per prior-rolled bucket ---
ax = axes[1]
ax.bar(dr_rolled.index.astype(str), dr_rolled['count'],
       color=PALETTE['neutral'], edgecolor='white')
ax.set_title('Episode volume by prior\nrolled count', fontweight='bold')
ax.set_xlabel('Prior episodes rolled past 30d')
ax.set_ylabel('Episode count')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1000:.0f}k'))

# --- cleared-within-24h history vs current outcome ---
ax = axes[2]
first_ep = df[df['prior_overdraw_count'] >= 2].copy()
first_ep['clear_rate_prior'] = first_ep['prior_cleared_within_24h_count'] / first_ep['prior_overdraw_count'].clip(lower=1)
first_ep['clear_rate_band'] = pd.cut(first_ep['clear_rate_prior'],
                                      bins=[0, 0.25, 0.5, 0.75, 1.001],
                                      labels=['0–25%', '25–50%', '50–75%', '75–100%'],
                                      include_lowest=True)
dr_clear = first_ep.groupby('clear_rate_band', observed=True)['defaulted_30d'].mean() * 100
bars = ax.bar(dr_clear.index, dr_clear.values, color=PALETTE['neutral'], edgecolor='white')
for bar, v in zip(bars, dr_clear.values):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.1,
            f'{v:.1f}%', ha='center', va='bottom', fontsize=10)
ax.set_title('Default rate by prior\n24h-clear rate', fontweight='bold')
ax.set_xlabel('Fraction of prior episodes cleared within 24h')
ax.set_ylabel('Current default rate (%)')

fig.suptitle('Repayment history features — behavioural track record matters',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

print('Prior rolled count → current default rate (r = 0.38 with default):')
print(dr_rolled[['mean_pct','count']].rename(columns={'mean_pct':'default_%','count':'episodes'}).round(1).to_string())

**This is the strongest behavioral-history signal in the dataset.**

- A borrower with **zero prior rollovers** defaults at 3.2% on their next episode — below the portfolio average
- A borrower with **one prior rollover** defaults at 29.6% — nearly 6× the portfolio average
- A borrower with **three prior rollovers** defaults at 53.6% — more likely to default than not

This near-monotonic escalation is consistent with a **behavioral persistence** effect: borrowers who structurally struggle to clear within 30 days keep struggling. The 24h-clear rate shows the mirror image — fast repayers stay fast repayers.

This is exactly the kind of signal that makes mobile-money repayment history valuable as a thin-file credit signal: it doesn't require a credit bureau; the repayment *behavior itself* is the history.

---
## 9. Feature Correlation & Multicollinearity

In [ ]:
import matplotlib.colors as mcolors

corr_matrix = df[model_features + ['defaulted_30d']].corr()

fig, axes = plt.subplots(1, 2, figsize=(17, 7))

# --- heatmap ---
ax = axes[0]
cmap = plt.cm.RdBu_r
im = ax.imshow(corr_matrix.values, cmap=cmap, vmin=-1, vmax=1, aspect='auto')
ax.set_xticks(range(len(corr_matrix.columns)))
ax.set_yticks(range(len(corr_matrix.columns)))
ax.set_xticklabels(corr_matrix.columns, rotation=45, ha='right', fontsize=8)
ax.set_yticklabels(corr_matrix.columns, fontsize=8)
plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
ax.set_title('Feature correlation matrix', fontweight='bold')

# --- bar chart: correlation with target ---
ax = axes[1]
target_corr = corr_matrix['defaulted_30d'].drop('defaulted_30d').sort_values()
colors_bar = [PALETTE['default'] if v > 0 else PALETTE['cleared'] for v in target_corr.values]
bars = ax.barh(range(len(target_corr)), target_corr.values, color=colors_bar, edgecolor='white')
ax.set_yticks(range(len(target_corr)))
ax.set_yticklabels(target_corr.index, fontsize=9)
ax.axvline(0, color='gray', linewidth=0.8)
ax.set_xlabel('Pearson correlation with defaulted_30d')
ax.set_title('Feature correlations with target\n(red=positive risk, green=protective)', fontweight='bold')
for bar, v in zip(bars, target_corr.values):
    x_pos = v + 0.01 if v >= 0 else v - 0.01
    ha = 'left' if v >= 0 else 'right'
    ax.text(x_pos, bar.get_y()+bar.get_height()/2, f'{v:.2f}', va='center', fontsize=8)

fig.suptitle('Correlation analysis', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print('Top 5 positive (risk-increasing) correlations with default:')
print(target_corr.sort_values(ascending=False).head(5).round(4).to_string())
print()
print('Top 5 negative (protective) correlations with default:')
print(target_corr.sort_values().head(5).round(4).to_string())

**Key observations:**

| Feature | Correlation with default | Direction | Business interpretation |
|---|---|---|---|
| `overdraw_to_inflow_ratio` | +0.70 | Risk ↑ | Draw large relative to inflow → can't repay quickly |
| `prior_rolled_past_30d_count` | +0.38 | Risk ↑ | Past defaults predict future defaults |
| `utilization_rate` | +0.23 | Risk ↑ | Drawing near limit → less buffer |
| `monthly_txn_count` | −0.22 | Protective | Active users have regular inflow to repay from |
| `avg_txn_value_kes` | −0.21 | Protective | Higher-value users have larger inflow buffers |

**Multicollinearity notes:**  
- `monthly_txn_count` and `avg_txn_value_kes` are moderately correlated with each other (~0.3), as both proxy for overall wallet activity. LightGBM handles this gracefully via split-based feature importance without the inflation problems that affect linear models.
- `prior_overdraw_count`, `prior_cleared_within_24h_count`, and `prior_rolled_past_30d_count` are naturally correlated (all depend on how many episodes a borrower has had). Together they paint a repayment history picture; no single one captures it fully.

---
## 10. Fairness Considerations

Alternative-data credit scoring carries a specific fairness risk: behavioral signals are correlated with socioeconomic position in ways that can produce disparate outcomes for vulnerable groups — **even without explicit protected-class data in the model**.

Two proxy groups are audited here and enforced as a CI/CD release gate:

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# --- proxy 1: tenure (thin file) ---
ax = axes[0]
df['thin_file'] = (df['tenure_months'] < 12).astype(int)
for col_label, group_label, col in [
    ('thin_file', 'Thin file\n(<12 months)', PALETTE['default']),
    (None, 'Established\n(≥12 months)', PALETTE['cleared'])
]:
    pass
groups = {0: 'Established (≥12mo)', 1: 'Thin file (<12mo)'}
colors_g = {0: PALETTE['cleared'], 1: PALETTE['default']}
for g, label in groups.items():
    sub = df[df['thin_file'] == g]
    ax.hist(sub['overdraw_to_inflow_ratio'].clip(upper=20), bins=40,
            density=True, alpha=0.5, color=colors_g[g], label=f'{label}\n(n={len(sub):,})')
ax.set_title('OIR distribution by tenure group\n(thin file vs. established)', fontweight='bold')
ax.set_xlabel('Overdraw-to-inflow ratio (capped at 20)')
ax.set_ylabel('Density')
ax.legend(fontsize=8)

# --- proxy 2: CRB flag ---
ax = axes[1]
for g, label in [(0, 'CRB clean'), (1, 'CRB flagged')]:
    sub = df[df['crb_flagged'] == g]
    ax.hist(sub['overdraw_to_inflow_ratio'].clip(upper=20), bins=40,
            density=True, alpha=0.5,
            color=PALETTE['cleared'] if g == 0 else PALETTE['default'],
            label=f'{label} (n={len(sub):,})')
ax.set_title('OIR distribution by CRB status', fontweight='bold')
ax.set_xlabel('Overdraw-to-inflow ratio (capped at 20)')
ax.set_ylabel('Density')
ax.legend(fontsize=9)

# --- four-fifths rule visualization ---
ax = axes[2]
import json, pathlib
try:
    fr = json.loads(pathlib.Path('../models/fairness_report.json').read_text())
    groups_f = list(fr.keys())
    ratios = [fr[g]['disparate_impact_ratio'] for g in groups_f]
    group_labels = ['Tenure proxy\n(thin file)', 'CRB flag proxy']
    bars = ax.bar(group_labels, ratios,
                  color=[PALETTE['cleared'] if r >= 0.8 else PALETTE['default'] for r in ratios],
                  width=0.4, edgecolor='white')
    ax.axhline(0.8, color=PALETTE['default'], linestyle='--', linewidth=2,
               label='4/5 rule threshold (0.80)')
    ax.axhline(1.0, color='gray', linestyle=':', linewidth=1, label='Parity (1.00)')
    for bar, v in zip(bars, ratios):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01,
                f'{v:.3f}\n✓ PASS', ha='center', va='bottom',
                fontsize=10, color=PALETTE['cleared'], fontweight='bold')
    ax.set_ylim(0, 1.3)
    ax.set_title('Four-fifths rule audit\n(model output, CI-enforced)', fontweight='bold')
    ax.set_ylabel('Disparate impact ratio')
    ax.legend(fontsize=8)
except FileNotFoundError:
    ax.text(0.5, 0.5, 'Run src/models/train.py\nto generate fairness_report.json',
            ha='center', va='center', transform=ax.transAxes)

fig.suptitle('Fairness audit — thin-file and CRB proxy groups', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

**Four-fifths (disparate impact) rule:**  
If the approval rate for a disadvantaged group is less than 80% of the approval rate for the reference group, the model fails the fairness gate and the CI build is blocked.

| Proxy group | Disadvantaged | Reference | Disparate impact ratio | Result |
|---|---|---|---|---|
| Tenure (thin file <12mo) | 87.9% approval | 86.8% approval | **1.013** | ✓ Pass |
| CRB flagged | 85.8% approval | 87.0% approval | **0.986** | ✓ Pass |

Both groups pass. Thin-file borrowers are not disadvantaged — their approval rate is marginally higher than established borrowers', which is consistent with the model using episode-level risk signals (OIR, utilization) rather than penalizing short tenure directly.

**CRB-flagged group note:** CRB-flagged borrowers show higher FPR (11.65% vs 9.92%, z-test p=0.011) and higher FNR (60.1% vs 53.4%). Both the DI ratio (0.986) and approval-rate gap are within legal thresholds, but the double adverse-impact pattern — wrong denials more often *and* missed defaults more often — warrants documented investigation before a production deployment. See the README Findings Interpretation section.

**Limitations of this audit:**  
These are *proxy* groups, not actual protected-class data. A real deployment would require auditing against gender, age, and geographic-disadvantage data subject to Kenya's Data Protection Act and the CBK Digital Credit Providers Act 2022.

---
## 11. From EDA to Model — Objectives Fulfilled

This section connects the EDA findings to the three project objectives and confirms each was addressed.

In [ ]:
import json, pathlib

try:
    metrics = json.loads(pathlib.Path('../models/metrics.json').read_text())
    dep = metrics["deployment_threshold"]
    cm = dep["confusion_matrix"]
    print('=== Trained model metrics (from models/metrics.json) ===')
    print(f'  AUC:                  {metrics["auc"]:.4f}')
    print(f'  PR-AUC:               {metrics["pr_auc"]:.4f}')
    print(f'  Brier score:          {metrics["brier_score"]:.4f}')
    print(f'  Decision threshold:   {dep["threshold"]:.6f}  ({dep["derivation"]})')
    print(f'  Precision @ thresh:   {dep["precision"]:.4f}')
    print(f'  Recall @ thresh:      {dep["recall"]:.4f}')
    print(f'  Flag rate @ thresh:   {dep["flag_rate"]:.4f}')
    print(f'  Confusion matrix:     TN={cm["tn"]:,}  FP={cm["fp"]:,}  FN={cm["fn"]:,}  TP={cm["tp"]:,}')
    tv = metrics["train_val_check"]
    print(f'  Train AUC:            {tv["train_auc"]:.4f}  Val AUC: {tv["val_auc"]:.4f}  Gap: {tv["gap"]:.4f}')
except FileNotFoundError:
    print('Run src/models/train.py first to generate metrics.json')

In [ ]:
try:
    sample_rc = json.loads(pathlib.Path('../models/sample_reason_codes.json').read_text())
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    for i, (ax, example) in enumerate(zip(axes, sample_rc[:2])):
        rc = example['reason_codes']
        feats = [r['feature'] for r in rc]
        shap_vals = [r['shap_value'] for r in rc]
        colors_rc = [PALETTE['default'] if v > 0 else PALETTE['cleared'] for v in shap_vals]
        ax.barh(feats[::-1], shap_vals[::-1], color=colors_rc[::-1], edgecolor='white')
        ax.axvline(0, color='gray', linewidth=0.8)
        ax.set_title(
            f"Example {i+1} — PD: {example['probability_of_default']:.4f}  Decision: {example['decision']}",
            fontweight='bold'
        )
        ax.set_xlabel('SHAP value (log-odds contribution)')
    
    fig.suptitle('SHAP adverse-action reason codes (per-prediction explanations)', fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.show()
except (FileNotFoundError, KeyError):
    print('Run src/models/train.py first to generate sample_reason_codes.json')

### Objective fulfilment summary

---

#### Objective 1 — Predict 30-day default using alternative behavioral data only

**EDA finding:** `overdraw_to_inflow_ratio` (r ≈ 0.35), `prior_rolled_past_30d_count`, and `utilization_rate` are strong discriminators using *only* behavioral signals — no income, employment, or bureau financial data.

**Model outcome:** AUC **0.7936**, PR-AUC **0.3019**. The DGP was designed for realism: a latent risk factor θ (not in the feature set) drives part of each borrower's default probability, and an idiosyncratic shock term (N(0, σ=2.0)) makes a subset of defaults structurally unpredictable from any feature. An oracle classifier with access to both observable features and the latent θ achieves AUC ≈ 0.81 — the trained model captures 98%+ of this ceiling. The remaining FNR (≈54%) is attributable to the irreducible shock, not to model or feature-set gaps. See the README **Findings Interpretation** section for the full oracle analysis.

---

#### Objective 2 — Identify and rank the dominant risk drivers, with per-prediction SHAP reason codes

**EDA finding:** OIR is the single strongest signal, followed by repayment history (`prior_rolled_past_30d_count`) and episode-level signals (utilization, draw amount). All are available at scoring time — no post-outcome leakage.

**Model outcome:** SHAP TreeExplainer generates per-prediction adverse-action reason codes (top-3 SHAP contributors per score). The API returns these alongside the probability and decision — satisfying the regulatory requirement for explainable automated lending decisions.

---

#### Objective 3 — Audit for proxy discrimination; enforce as CI/CD gate

**EDA finding:** Thin-file (<12 months tenure) and CRB-flagged borrowers show modestly elevated default rates but are not disproportionately disadvantaged by the model on the four-fifths rule.

**Model outcome:** The GitHub Actions CI pipeline fails the build if either proxy group drops below 0.80 disparate impact ratio. The current model passes at **1.013** (tenure) and **0.986** (CRB flag). However, CRB-flagged borrowers show a double adverse-impact pattern (higher FPR *and* higher FNR) that must be investigated before any production deployment — see the README for the composition-effect hypothesis and production-readiness assessment.

---

## Appendix: System architecture

```
data/raw/overdraft_lending_data.csv  (190k episodes, committed static data)
         │
         ▼
src/features/build_features.py       validation gate + feature selection
         │
         ▼
src/models/train.py                  LightGBM training + SHAP + fairness audit
         │                           ──► models/{model.txt, metrics.json,
         │                                        fairness_report.json,
         │                                        feature_importance.png}
         ▼
src/api/main.py (FastAPI)            /score  /health  /model-info
         │
         ▼
Dockerfile (python:3.11-slim)        uvicorn serving, libgomp resolved via
         │                           LD_LIBRARY_PATH (scikit-learn bundled OpenMP)
         ▼
infra/AWS_DEPLOYMENT.md              Lambda container image + API Gateway
                                     + S3 + EventBridge drift monitoring
.github/workflows/ci.yml             tests → build features → train
                                     → fairness gate → AUC gate → Docker build
```

**Tech stack:** Python · LightGBM · SHAP · FastAPI · Pydantic · Docker · AWS Lambda · API Gateway · S3 · EventBridge · GitHub Actions · pytest